# Project 54 — Local Filesystem Agent
## Search, Summarize & Organize Files with LLM-Guided Tools

**Stack:** LangChain · Ollama · pathlib · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Core LLM & Tool Definitions

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pathlib import Path
import os, json

llm = ChatOllama(model="qwen3:8b", temperature=0.1)

@tool
def list_files(directory: str) -> str:
    """List files and directories in the given path."""
    p = Path(directory)
    if not p.exists():
        return f"Directory not found: {directory}"
    entries = sorted(p.iterdir())[:30]
    lines = []
    for e in entries:
        kind = "[DIR] " if e.is_dir() else "[FILE]"
        size = e.stat().st_size if e.is_file() else 0
        lines.append(f"{kind} {e.name:<40} {size:>8,} bytes")
    return "\n".join(lines) or "(empty)"

@tool
def read_file_content(filepath: str) -> str:
    """Read the first 1000 characters of a text file."""
    try:
        return Path(filepath).read_text(encoding="utf-8", errors="replace")[:1000]
    except Exception as e:
        return f"Error: {e}"

@tool
def file_stats(filepath: str) -> str:
    """Return file metadata: size, extension, line count."""
    p = Path(filepath)
    if not p.exists():
        return "File not found"
    stat = p.stat()
    try:
        lines = len(p.read_text(encoding="utf-8", errors="replace").splitlines())
    except Exception:
        lines = -1
    return json.dumps({"name": p.name, "size_bytes": stat.st_size,
                        "extension": p.suffix, "lines": lines})

tools = [list_files, read_file_content, file_stats]
print(f"Tools ready: {[t.name for t in tools]}")

## Step 2 — Create a Sample Workspace

In [ ]:
# Build a realistic tiny workspace to explore
ws = Path("sample_workspace")
ws.mkdir(exist_ok=True)
(ws / "src").mkdir(exist_ok=True)
(ws / "docs").mkdir(exist_ok=True)
(ws / "data").mkdir(exist_ok=True)

(ws / "README.md").write_text("# Demo Project\nA sample project for filesystem agent demo.")
(ws / "src" / "main.py").write_text(
    'import os\n\ndef main():\n    print("Hello!")\n\nif __name__ == "__main__":\n    main()\n')
(ws / "src" / "utils.py").write_text(
    'def add(a, b):\n    return a + b\n\ndef multiply(a, b):\n    return a * b\n')
(ws / "docs" / "setup.md").write_text("## Setup\n1. Install Python 3.10+\n2. Run `pip install -r req.txt`")
(ws / "data" / "config.json").write_text('{"db_host": "localhost", "port": 5432, "debug": true}')
(ws / "requirements.txt").write_text("langchain>=0.2\nollama\nchromadb\n")
print("Sample workspace created:")
print(list_files.invoke("sample_workspace"))

## Step 3 — Tool-Calling Research Loop

In [ ]:
def agent_research(question, workspace="sample_workspace"):
    """Multi-step: list → read relevant files → answer with context."""
    # Step 1 — inventory
    listing = list_files.invoke(workspace)

    # Step 2 — decide which files to read
    pick_prompt = ChatPromptTemplate.from_messages([
        ("system", "Given a file listing, pick the files most relevant to the question. "
                   "Return ONLY a JSON list of relative paths."),
        ("human", "Listing:\n{listing}\n\nQuestion: {question}")
    ])
    pick_chain = pick_prompt | llm | StrOutputParser()
    raw = pick_chain.invoke({"listing": listing, "question": question})
    # parse list from response
    import re
    picks = re.findall(r'[\w/\-\.]+\.[a-z]+', raw)
    print(f"  Files selected: {picks}")

    # Step 3 — gather content
    contents = {}
    for rel in picks[:5]:
        full = str(Path(workspace) / rel)
        contents[rel] = read_file_content.invoke(full)

    # Step 4 — answer
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a filesystem analyst. Use the file contents to answer."),
        ("human", "Question: {question}\n\nFile contents:\n{contents}")
    ])
    answer_chain = answer_prompt | llm | StrOutputParser()
    return answer_chain.invoke({"question": question,
                                 "contents": json.dumps(contents, indent=2)[:3000]})

print("Agent research loop ready")

## Step 4 — Run Agent Queries

In [ ]:
queries = [
    "What does this project do?",
    "What dependencies are required?",
    "Summarize the Python source code",
    "What is the database configuration?",
]

for q in queries:
    print(f"\nQ: {q}")
    answer = agent_research(q)
    print(f"A: {answer[:300]}")
    print("-" * 60)

## Step 5 — File Organization Suggestions

In [ ]:
from pydantic import BaseModel, Field

class OrgSuggestion(BaseModel):
    current_issues: list[str]
    recommended_structure: list[str]
    files_to_move: list[str]
    files_to_create: list[str]

organizer = llm.with_structured_output(OrgSuggestion)

listing = list_files.invoke("sample_workspace")
# also get sub-dirs
for sub in ["src", "docs", "data"]:
    listing += "\n--- " + sub + " ---\n"
    listing += list_files.invoke(f"sample_workspace/{sub}")

suggestion = organizer.invoke(
    f"Analyze this project structure and suggest improvements:\n{listing}"
)
print("ORGANIZATION REPORT")
print("=" * 50)
print(f"Issues: {suggestion.current_issues}")
print(f"Recommended structure: {suggestion.recommended_structure}")
print(f"Files to move: {suggestion.files_to_move}")
print(f"Missing files: {suggestion.files_to_create}")

## What You Learned
- **Tool-based file exploration** with pathlib wrappers
- **Multi-step agent loop**: list → pick → read → answer
- **Structured organization suggestions** via Pydantic